# 04 — Hospital Efficiency (RQ2)

**Research Question:** What does an efficient hospital look like?

Not all hospitals treating kidney stones perform equally. We quantify variation in LOS,
separate structural from operational effects, identify overperformers, and profile what
makes them different.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor

from shared import (
    load_kidney, load_hospital_tags, load_cnes_enriched,
    setup_plot_style, save_plot, save_metrics, CATEGORY_MAP,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
cnes = load_cnes_enriched()
print(f"Loaded {len(kidney):,} admissions, {len(tags)} hospitals, {len(cnes):,} CNES facilities")

Loaded 206,500 admissions, 510 hospitals, 109,849 CNES facilities


## 1. Hospital vs Procedure Effect on LOS

Which matters more: *which hospital* you go to, or *which procedure* you receive?

In [2]:
# Hospital effect: mean LOS per hospital
hosp_los = kidney.groupby("CNES")["DIAS_PERM"].mean()
hospital_effect = hosp_los.std()

# Procedure effect: mean LOS per procedure
proc_los = kidney.groupby("proc_category")["DIAS_PERM"].mean()
procedure_effect = proc_los.std()

print(f"Hospital effect (std of hospital mean LOS): {hospital_effect:.2f} days")
print(f"Procedure effect (std of category mean LOS): {procedure_effect:.2f} days")
print(f"Ratio: hospital/procedure = {hospital_effect/procedure_effect:.1f}x")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(hosp_los, bins=40, color="#2563EB", edgecolor="white", alpha=0.8)
axes[0].axvline(hosp_los.median(), color="red", linestyle="--",
                label=f"Median: {hosp_los.median():.1f}d")
axes[0].set_title("Distribution of Hospital Mean LOS")
axes[0].set_xlabel("Mean LOS (days)")
axes[0].set_ylabel("Number of hospitals")
axes[0].legend()

proc_los.sort_values().plot.barh(ax=axes[1], color="#059669")
axes[1].set_title("Mean LOS by Procedure Category")
axes[1].set_xlabel("Days")
axes[1].invert_yaxis()

fig.suptitle("Hospital vs Procedure Effect on LOS", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "effect_comparison", prefix="04")

Hospital effect (std of hospital mean LOS): 1.90 days
Procedure effect (std of category mean LOS): 1.02 days
Ratio: hospital/procedure = 1.9x


  Saved: 04_effect_comparison.png


## 2. Fair Comparison Within Peer Groups

Hospitals must be compared within their comparability group (facility type + admission profile + case mix).

In [3]:
# LOS variation within the largest peer groups
top_groups = tags["comparability_group"].value_counts().head(6).index

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, group in zip(axes.flat, top_groups):
    group_tags = tags[tags["comparability_group"] == group]
    los_vals = group_tags["mean_los"].dropna()
    ax.hist(los_vals, bins=20, color="#2563EB", edgecolor="white", alpha=0.8)
    ax.axvline(los_vals.median(), color="red", linestyle="--")
    short_name = group.replace("__", "\n")
    ax.set_title(f"{short_name}\n(n={len(group_tags)})", fontsize=9)
    ax.set_xlabel("Mean LOS")

fig.suptitle("LOS Variation Within Peer Groups", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "peer_group_los", prefix="04")

  Saved: 04_peer_group_los.png


## 2b. Within-Group Variation Quantification

How much LOS variation exists within peer groups vs between them? This is an ANOVA-style
decomposition that tells us whether the variation is due to hospital practice or group membership.

In [4]:
min_group_size = 5
group_stats = tags.groupby("comparability_group").agg(
    n_hospitals=pd.NamedAgg(column="CNES", aggfunc="count"),
    mean_los=pd.NamedAgg(column="mean_los", aggfunc="mean"),
    std_los=pd.NamedAgg(column="mean_los", aggfunc="std"),
    median_los=pd.NamedAgg(column="mean_los", aggfunc="median"),
    min_los=pd.NamedAgg(column="mean_los", aggfunc="min"),
    max_los=pd.NamedAgg(column="mean_los", aggfunc="max"),
).dropna()
group_stats["range_los"] = group_stats["max_los"] - group_stats["min_los"]
group_stats["cv"] = group_stats["std_los"] / group_stats["mean_los"]

large_groups = group_stats[group_stats["n_hospitals"] >= min_group_size].sort_values("n_hospitals", ascending=False)
print(f"Peer groups with >={min_group_size} hospitals: {len(large_groups)}")
print(f"\n=== WITHIN-GROUP LOS VARIATION ===")
print(large_groups[["n_hospitals", "mean_los", "std_los", "cv", "min_los", "max_los", "range_los"]].head(10).to_string())

# Kruskal-Wallis: are groups different from each other?
groups_for_test = []
for grp_name, grp_tags in tags.groupby("comparability_group"):
    if len(grp_tags) >= min_group_size:
        vals = grp_tags["mean_los"].dropna().values
        if len(vals) >= min_group_size:
            groups_for_test.append(vals)

if len(groups_for_test) >= 2:
    h_stat, h_p = stats.kruskal(*groups_for_test)
    print(f"\nKruskal-Wallis H test (between-group differences): H={h_stat:.1f}, p={h_p:.2e}")
    print("  → Groups are significantly different" if h_p < 0.05 else "  → No significant difference")

# Within the largest group: how much variation?
largest_group = large_groups.index[0]
lg_hospitals = tags[tags["comparability_group"] == largest_group]
print(f"\nLargest group: {largest_group}")
print(f"  n={len(lg_hospitals)}, LOS range: {lg_hospitals['mean_los'].min():.1f}–{lg_hospitals['mean_los'].max():.1f}d")
print(f"  If worst matched median: {(lg_hospitals['mean_los'].max() - lg_hospitals['mean_los'].median()):.1f}d improvement potential")

Peer groups with >=5 hospitals: 19

=== WITHIN-GROUP LOS VARIATION ===
                                                              n_hospitals  mean_los   std_los        cv   min_los    max_los  range_los
comparability_group                                                                                                                    
hospital_geral__emergency_dominant__diagnostic_heavy                  244  3.071016  1.532626  0.499062  0.500000  12.000000  11.500000
hospital_geral__emergency_dominant__mixed_procedures                   47  2.514780  1.403256  0.558003  0.000000   9.363636   9.363636
hospital_geral__mixed__mixed_procedures                                27  2.316501  0.764236  0.329910  1.500000   5.125000   3.625000
hospital_geral__elective_dominant__surgical_center                     23  1.607873  0.951896  0.592022  0.000000   3.827715   3.827715
hospital_geral__emergency_dominant__surgical_center                    21  2.505010  0.810082  0.323385  1.431250

## 3. Overperformance Scoring

Separate structural factors (what a hospital *is*) from operational performance (how well it *executes*).
A hospital that achieves lower LOS than expected given its structure is an overperformer.

In [5]:
# Merge hospital tags with CNES enriched features
hosp_features = tags.merge(
    cnes[["CNES", "legal_nature", "active_committees", "total_equipment",
          "total_beds", "surgical_beds", "icu_beds", "doctors", "nurses",
          "doctors_per_bed", "nurses_per_bed"]].drop_duplicates(subset=["CNES"]),
    on="CNES", how="left"
)

# Structural features for predicting expected LOS
struct_features = ["elective_rate", "n_admissions"]
optional_features = ["total_beds", "surgical_beds", "icu_beds",
                     "doctors_per_bed", "nurses_per_bed", "active_committees",
                     "total_equipment"]
struct_features += [f for f in optional_features if f in hosp_features.columns
                    and hosp_features[f].notna().sum() > len(hosp_features) * 0.3]

# Encode categorical
for cat_col in ["facility_type", "admission_profile", "case_mix_profile"]:
    if cat_col in hosp_features.columns:
        dummies = pd.get_dummies(hosp_features[cat_col], prefix=cat_col, drop_first=True)
        hosp_features = pd.concat([hosp_features, dummies], axis=1)
        struct_features += list(dummies.columns)

# Filter to hospitals with enough data
min_admissions = 20
modeling_set = hosp_features[hosp_features["n_admissions"] >= min_admissions].copy()
modeling_set = modeling_set.dropna(subset=["mean_los"])
print(f"Modeling set: {len(modeling_set)} hospitals (>={min_admissions} admissions)")

available_features = [f for f in struct_features if f in modeling_set.columns]
X = modeling_set[available_features].fillna(0)
y = modeling_set["mean_los"]

model = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
model.fit(X, y)
modeling_set["expected_los"] = model.predict(X)
modeling_set["overperformance"] = modeling_set["expected_los"] - modeling_set["mean_los"]

r2 = model.score(X, y)
print(f"R² (structural model): {r2:.3f}")
print(f"Mean overperformance: {modeling_set['overperformance'].mean():.2f}d")
print(f"Std overperformance: {modeling_set['overperformance'].std():.2f}d")

Modeling set: 313 hospitals (>=20 admissions)
R² (structural model): 0.944
Mean overperformance: 0.00d
Std overperformance: 0.42d


In [6]:
# Feature importance
importances = pd.Series(model.feature_importances_, index=available_features).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

importances.tail(15).plot.barh(ax=axes[0], color="#2563EB")
axes[0].set_title("Feature Importance (Structural Model)")
axes[0].set_xlabel("Importance")

axes[1].scatter(modeling_set["expected_los"], modeling_set["mean_los"],
               c=modeling_set["overperformance"], cmap="RdYlGn", s=40, alpha=0.7)
lims = [0, max(modeling_set["expected_los"].max(), modeling_set["mean_los"].max()) + 1]
axes[1].plot(lims, lims, "--", color="gray", linewidth=0.8)
axes[1].set_title("Expected vs Actual LOS")
axes[1].set_xlabel("Expected LOS (structural)")
axes[1].set_ylabel("Actual Mean LOS")
axes[1].annotate("Overperformers\n(below line)", xy=(0.7, 0.2), xycoords="axes fraction",
                fontsize=10, color="green")
axes[1].annotate("Underperformers\n(above line)", xy=(0.1, 0.8), xycoords="axes fraction",
                fontsize=10, color="red")

fig.suptitle("Overperformance Model", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "overperformance_model", prefix="04")

  Saved: 04_overperformance_model.png


## 4. Profile of Top Performers

In [7]:
top_n = 10
top_performers = modeling_set.nlargest(top_n, "overperformance")
bottom_performers = modeling_set.nsmallest(top_n, "overperformance")

compare_cols = ["CNES", "n_admissions", "mean_los", "expected_los", "overperformance",
                "elective_rate", "mortality_rate", "facility_type", "admission_profile"]
available_compare = [c for c in compare_cols if c in modeling_set.columns]

print(f"=== TOP {top_n} OVERPERFORMERS ===")
print(top_performers[available_compare].to_string(index=False))
print(f"\n=== BOTTOM {top_n} UNDERPERFORMERS ===")
print(bottom_performers[available_compare].to_string(index=False))

# Comparison chart
metrics_to_compare = ["mean_los", "elective_rate", "mortality_rate"]
available_metrics = [m for m in metrics_to_compare if m in modeling_set.columns]

fig, axes = plt.subplots(1, len(available_metrics), figsize=(5 * len(available_metrics), 5))
if len(available_metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, available_metrics):
    data = pd.DataFrame({
        "Top 10": [top_performers[metric].mean()],
        "Bottom 10": [bottom_performers[metric].mean()],
        "All": [modeling_set[metric].mean()],
    }).T
    data.plot.bar(ax=ax, legend=False, color=["#059669", "#DC2626", "#6B7280"])
    ax.set_title(metric.replace("_", " ").title())
    ax.tick_params(axis="x", rotation=0)

fig.suptitle("Top vs Bottom Performers", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "top_vs_bottom", prefix="04")

=== TOP 10 OVERPERFORMERS ===
   CNES  n_admissions  mean_los  expected_los  overperformance  elective_rate  mortality_rate          facility_type  admission_profile
2077671           243  2.226337      3.265045         1.038707       0.004115        0.000000         hospital_geral emergency_dominant
2082888           640  1.431250      2.280583         0.849333       0.221875        0.006250         hospital_geral emergency_dominant
2747871            89  2.146067      2.981554         0.835487       0.022472        0.000000         hospital_geral emergency_dominant
2077558            45  1.688889      2.520300         0.831412       0.000000        0.000000         hospital_geral emergency_dominant
7947984            40  1.625000      2.445035         0.820035       0.050000        0.000000         hospital_geral emergency_dominant
7373465          1492  1.416220      2.234455         0.818235       0.677614        0.003351         hospital_geral  elective_dominant
0008052           

## 5. Ureteroscopy Hospital Variation

For hospitals performing the same modern procedure, how much does LOS still vary?

In [8]:
uretero = kidney[kidney["has_new_proc"] == 1]
uretero_hosp = uretero.groupby("CNES").agg(
    n=pd.NamedAgg(column="CNES", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
).reset_index()
uretero_hosp = uretero_hosp[uretero_hosp["n"] >= 10]

fig, ax = plt.subplots(figsize=(12, 6))
uretero_sorted = uretero_hosp.sort_values("mean_los")
colors = ["#059669" if x <= uretero_sorted["mean_los"].median() else "#DC2626"
          for x in uretero_sorted["mean_los"]]
ax.barh(range(len(uretero_sorted)), uretero_sorted["mean_los"], color=colors)
ax.axvline(uretero_sorted["mean_los"].median(), color="black", linestyle="--",
           label=f"Median: {uretero_sorted['mean_los'].median():.1f}d")
ax.set_title(f"Ureteroscopy Mean LOS by Hospital (n>={10}, {len(uretero_sorted)} hospitals)")
ax.set_xlabel("Mean LOS (days)")
ax.set_yticks([])
ax.legend()
fig.tight_layout()
save_plot(fig, "ureteroscopy_variation", prefix="04")

  Saved: 04_ureteroscopy_variation.png


## Summary Metrics

In [9]:
top_op_mean = top_performers["overperformance"].mean()
bot_op_mean = bottom_performers["overperformance"].mean()
u_top_bot, p_top_bot = stats.mannwhitneyu(
    top_performers["mean_los"], bottom_performers["mean_los"], alternative="less"
)

metrics = {
    "hospital_effect_std": round(float(hospital_effect), 2),
    "procedure_effect_std": round(float(procedure_effect), 2),
    "effect_ratio": round(float(hospital_effect / procedure_effect), 1),
    "structural_model_r2": round(float(r2), 3),
    "hospitals_modeled": int(len(modeling_set)),
    "peer_groups_with_5plus": int(len(large_groups)),
    "largest_group": str(largest_group),
    "largest_group_n": int(len(lg_hospitals)),
    "largest_group_los_range": f"{lg_hospitals['mean_los'].min():.1f}-{lg_hospitals['mean_los'].max():.1f}",
    "top10_mean_los": round(float(top_performers["mean_los"].mean()), 2),
    "top10_mean_overperf": round(float(top_op_mean), 2),
    "bottom10_mean_los": round(float(bottom_performers["mean_los"].mean()), 2),
    "bottom10_mean_overperf": round(float(bot_op_mean), 2),
    "top_vs_bottom_p": float(p_top_bot),
    "ureteroscopy_hospitals": int(len(uretero_sorted)),
    "ureteroscopy_los_range": f"{uretero_sorted['mean_los'].min():.1f}-{uretero_sorted['mean_los'].max():.1f}",
}
save_metrics(metrics, "04_hospital_efficiency")

modeling_set[["CNES", "n_admissions", "mean_los", "expected_los", "overperformance",
              "facility_type", "admission_profile", "case_mix_profile"]].to_parquet(
    save_metrics.__defaults__[0].parent / "hospital_ranking.parquet", index=False
)

print("\n=== HOSPITAL EFFICIENCY ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 04_hospital_efficiency.json

=== HOSPITAL EFFICIENCY ===
  hospital_effect_std: 1.9
  procedure_effect_std: 1.02
  effect_ratio: 1.9
  structural_model_r2: 0.944
  hospitals_modeled: 313
  peer_groups_with_5plus: 19
  largest_group: hospital_geral__emergency_dominant__diagnostic_heavy
  largest_group_n: 244
  largest_group_los_range: 0.5-12.0
  top10_mean_los: 1.77
  top10_mean_overperf: 0.82
  bottom10_mean_los: 4.75
  bottom10_mean_overperf: -1.05
  top_vs_bottom_p: 9.133589555477501e-05
  ureteroscopy_hospitals: 113
  ureteroscopy_los_range: 0.0-6.7


---

## 6. Deep Dive: Why Are Some Hospitals Better?

The sections above establish **that** hospitals differ enormously within peer groups. This section investigates **why**, using three independent axes:

1. **Patient mix control** — Are top/bottom hospitals treating the same patients? (confounding check)
2. **Structural profile** — Do they differ in resources, governance, or legal nature?
3. **Operational fingerprint** — Do they differ in process choices (same-day rate, specialization, long-stay management)?

Plus same-city case studies and legal nature analysis.

In [10]:
# === 6a. Patient Mix Control ===
# Compare age, sub-diagnosis, severity proxies between top/bottom per group

pat = kidney.groupby("CNES").agg(
    n=("CNES", "count"),
    mean_age=("age", "mean"),
    pct_elderly=("age", lambda x: (x >= 65).mean()),
    pct_male=("is_male", "mean"),
    pct_n200=("DIAG_PRINC", lambda x: (x == "N200").mean()),
    pct_n201=("DIAG_PRINC", lambda x: (x == "N201").mean()),
    uti_rate=("MARCA_UTI", lambda x: (x != "00").mean() if x.dtype == object else 0),
    mean_los=("DIAS_PERM", "mean"),
).reset_index()
pat = pat[pat["n"] >= 20]
pat = pat.merge(tags[["CNES", "comparability_group"]], on="CNES", how="left")

grp_counts = pat["comparability_group"].value_counts()
large_groups = grp_counts[grp_counts >= 10].index.tolist()

mix_rows = []
for grp in large_groups:
    g = pat[pat["comparability_group"] == grp].sort_values("mean_los")
    n_show = min(10, len(g))
    top, bot = g.head(n_show), g.tail(n_show)
    grp_label = " · ".join(grp.split("__")[1:])
    
    for col, label in [("mean_age", "Idade"), ("pct_elderly", "% Idosos"),
                        ("pct_male", "% Masculino"), ("pct_n200", "% N20.0"),
                        ("pct_n201", "% N20.1"), ("uti_rate", "% UTI")]:
        _, p = stats.mannwhitneyu(top[col], bot[col], alternative="two-sided")
        mix_rows.append({
            "Grupo": grp_label, "Variável": label,
            "Top 10": f"{top[col].mean():.3f}", "Bottom 10": f"{bot[col].mean():.3f}",
            "p-value": p, "Significativo": "Sim" if p < 0.05 else "Não"
        })

mix_df = pd.DataFrame(mix_rows)
sig_count = mix_df[mix_df["Significativo"] == "Sim"]
age_sig = mix_df[(mix_df["Variável"] == "Idade") & (mix_df["Significativo"] == "Sim")]

print("PATIENT MIX: Top 10 vs Bottom 10 por grupo")
print(f"  Variáveis significativas: {len(sig_count)} de {len(mix_df)}")
print(f"  Idade significativa em: {len(age_sig)}/{len(large_groups)} grupos")
print(f"  UTI significativa em: {len(mix_df[(mix_df['Variável'] == '% UTI') & (mix_df['Significativo'] == 'Sim')])}/{len(large_groups)} grupos")
print()

# Show only significant results
if len(sig_count):
    print("Diferenças significativas (p < 0.05):")
    print(sig_count[["Grupo", "Variável", "Top 10", "Bottom 10", "p-value"]].to_string(index=False))
else:
    print("Nenhuma diferença significativa no mix de pacientes.")

print("\nVEREDICTO: Idade, sexo e sub-diagnóstico são SIMILARES entre top e bottom.")
print("A diferença de LOS NÃO é explicada pelo mix de pacientes.")
print("Exceção: UTI rate é significativamente maior nos bottom (sinal de complicações).")

PATIENT MIX: Top 10 vs Bottom 10 por grupo
  Variáveis significativas: 4 de 42
  Idade significativa em: 0/7 grupos
  UTI significativa em: 3/7 grupos

Diferenças significativas (p < 0.05):
                                Grupo    Variável Top 10 Bottom 10  p-value
emergency_dominant · diagnostic_heavy       % UTI  0.001     0.033 0.000928
  elective_dominant · surgical_center % Masculino  0.471     0.431 0.031209
  elective_dominant · surgical_center       % UTI  0.005     0.046 0.000329
 emergency_dominant · surgical_center       % UTI  0.012     0.035 0.017090

VEREDICTO: Idade, sexo e sub-diagnóstico são SIMILARES entre top e bottom.
A diferença de LOS NÃO é explicada pelo mix de pacientes.
Exceção: UTI rate é significativamente maior nos bottom (sinal de complicações).


### 6b. Structural Profile — "What Do They Have?"

In [11]:
# === 6b. Structural Profile ===
hosp_struct = kidney.groupby("CNES").agg(
    n=("CNES", "count"), mean_los=("DIAS_PERM", "mean"),
).reset_index()
hosp_struct = hosp_struct[hosp_struct["n"] >= 20]
hosp_struct = hosp_struct.merge(tags[["CNES", "comparability_group"]], on="CNES", how="left")
hosp_struct = hosp_struct.merge(
    cnes[["CNES", "total_beds", "surgical_beds", "icu_beds", "sus_bed_fraction",
          "total_equipment", "ct_scanners", "equip_utilization", "active_committees",
          "legal_nature"]], on="CNES", how="left")

struct_cols = ["total_beds", "total_equipment", "active_committees", "ct_scanners"]
struct_rows = []

for grp in large_groups:
    g = hosp_struct[hosp_struct["comparability_group"] == grp].sort_values("mean_los")
    n_show = min(10, len(g))
    top, bot = g.head(n_show), g.tail(n_show)
    grp_label = " · ".join(grp.split("__")[1:])
    
    for col in struct_cols:
        t_vals, b_vals = top[col].dropna(), bot[col].dropna()
        if len(t_vals) < 3 or len(b_vals) < 3:
            continue
        _, p = stats.mannwhitneyu(t_vals, b_vals, alternative="two-sided")
        struct_rows.append({
            "Grupo": grp_label, "Recurso": col,
            "Top 10 (mediana)": t_vals.median(),
            "Bottom 10 (mediana)": b_vals.median(),
            "p-value": p
        })

struct_df = pd.DataFrame(struct_rows)
sig_struct = struct_df[struct_df["p-value"] < 0.05]

print("STRUCTURAL PROFILE: Top 10 vs Bottom 10")
print(f"  Significativas: {len(sig_struct)} de {len(struct_df)}\n")
print(sig_struct.to_string(index=False))

# Visualization: beds top vs bottom
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Aggregate across all groups
all_top_beds, all_bot_beds = [], []
for grp in large_groups:
    g = hosp_struct[hosp_struct["comparability_group"] == grp].sort_values("mean_los")
    n_show = min(10, len(g))
    all_top_beds.extend(g.head(n_show)["total_beds"].dropna().tolist())
    all_bot_beds.extend(g.tail(n_show)["total_beds"].dropna().tolist())

axes[0].hist(all_top_beds, bins=20, alpha=0.7, label=f"Top (n={len(all_top_beds)})", color="#2ecc71")
axes[0].hist(all_bot_beds, bins=20, alpha=0.7, label=f"Bottom (n={len(all_bot_beds)})", color="#e74c3c")
axes[0].set_xlabel("Total de Leitos")
axes[0].set_ylabel("Hospitais")
axes[0].set_title("Distribuição de Leitos: Top vs Bottom")
axes[0].legend()
axes[0].axvline(np.median(all_top_beds), color="#2ecc71", ls="--", lw=2)
axes[0].axvline(np.median(all_bot_beds), color="#e74c3c", ls="--", lw=2)

# Legal nature pie charts
all_top_ln, all_bot_ln = [], []
for grp in large_groups:
    g = hosp_struct[hosp_struct["comparability_group"] == grp].sort_values("mean_los")
    n_show = min(10, len(g))
    all_top_ln.extend(g.head(n_show)["legal_nature"].dropna().tolist())
    all_bot_ln.extend(g.tail(n_show)["legal_nature"].dropna().tolist())

ln_labels = {"assoc_privada": "Santa Casa", "public": "Público",
             "fundacao_privada": "Fundação", "private": "Privado"}
top_ln = pd.Series(all_top_ln).map(ln_labels).value_counts()
bot_ln = pd.Series(all_bot_ln).map(ln_labels).value_counts()

x = np.arange(len(top_ln.index.union(bot_ln.index)))
all_cats = sorted(top_ln.index.union(bot_ln.index))
w = 0.35
axes[1].bar(x - w/2, [top_ln.get(c, 0) for c in all_cats], w, label="Top 10", color="#2ecc71")
axes[1].bar(x + w/2, [bot_ln.get(c, 0) for c in all_cats], w, label="Bottom 10", color="#e74c3c")
axes[1].set_xticks(x)
axes[1].set_xticklabels(all_cats)
axes[1].set_ylabel("Hospitais")
axes[1].set_title("Natureza Jurídica: Top vs Bottom")
axes[1].legend()

plt.tight_layout()
save_plot(fig, "04_structural_top_vs_bottom")
plt.show()

_, p_beds = stats.mannwhitneyu(all_top_beds, all_bot_beds, alternative="two-sided")
print(f"\nLeitos: top mediana={np.median(all_top_beds):.0f} vs bottom mediana={np.median(all_bot_beds):.0f} (p={p_beds:.6f})")
print(f"Natureza jurídica top: {dict(top_ln)}")
print(f"Natureza jurídica bottom: {dict(bot_ln)}")
print("\nVEREDICTO: Hospitais MENORES são mais eficientes. Santa Casas dominam o top.")

STRUCTURAL PROFILE: Top 10 vs Bottom 10
  Significativas: 10 de 28

                                Grupo           Recurso  Top 10 (mediana)  Bottom 10 (mediana)  p-value
emergency_dominant · diagnostic_heavy        total_beds              47.0                181.0 0.007285
emergency_dominant · diagnostic_heavy   total_equipment              20.0                 29.0 0.037276
emergency_dominant · diagnostic_heavy active_committees               5.0                  9.5 0.004602
             mixed · mixed_procedures        total_beds             164.0                359.0 0.004586
             mixed · mixed_procedures   total_equipment              36.0                 47.5 0.041098
  elective_dominant · surgical_center        total_beds              61.5                174.0 0.000504
  elective_dominant · surgical_center   total_equipment              26.5                 37.5 0.028306
 emergency_dominant · surgical_center        total_beds              79.5                277.5 0.001

  Saved: 04_structural_top_vs_bottom.png

Leitos: top mediana=110 vs bottom mediana=212 (p=0.000000)
Natureza jurídica top: {'Santa Casa': np.int64(43), 'Público': np.int64(22), 'Fundação': np.int64(3), 'Privado': np.int64(2)}
Natureza jurídica bottom: {'Público': np.int64(38), 'Santa Casa': np.int64(27), 'Fundação': np.int64(5)}

VEREDICTO: Hospitais MENORES são mais eficientes. Santa Casas dominam o top.


### 6c. Operational Fingerprint — "How Do They Operate?"

In [12]:
# === 6c. Operational Fingerprint ===
ops = kidney.groupby("CNES").agg(
    n=("CNES", "count"),
    mean_los=("DIAS_PERM", "mean"),
    sameday_rate=("DIAS_PERM", lambda x: (x == 0).mean()),
    longstay_rate=("DIAS_PERM", lambda x: (x > 7).mean()),
    has_uretero=("has_new_proc", "max"),
    migration_rate=("migrated", "mean"),
    uti_rate=("MARCA_UTI", lambda x: (x != "00").mean() if x.dtype == object else 0),
    proc_diversity=("PROC_REA", "nunique"),
).reset_index()
ops = ops[ops["n"] >= 20]
ops = ops.merge(tags[["CNES", "comparability_group"]], on="CNES", how="left")

op_cols = ["sameday_rate", "longstay_rate", "migration_rate", "uti_rate", "proc_diversity"]

# Spearman correlations within each group
corr_rows = []
for grp in large_groups:
    g = ops[ops["comparability_group"] == grp].copy()
    g["los_rank"] = g["mean_los"].rank()
    grp_label = " · ".join(grp.split("__")[1:])
    for col in op_cols:
        rho, p = stats.spearmanr(g[col], g["los_rank"], nan_policy="omit")
        corr_rows.append({"Grupo": grp_label, "Métrica": col, "ρ": rho, "p": p, "n": len(g)})

corr_df = pd.DataFrame(corr_rows)
sig_corr = corr_df[corr_df["p"] < 0.05].sort_values("ρ", key=abs, ascending=False)

print("SPEARMAN ρ: métrica operacional vs ranking de LOS (dentro do grupo)")
print(f"Correlações significativas: {len(sig_corr)} de {len(corr_df)}\n")
print(sig_corr.to_string(index=False))

# Visualization: top vs bottom operational metrics (aggregated)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

all_top_sd, all_bot_sd = [], []
all_top_ls, all_bot_ls = [], []
all_top_div, all_bot_div = [], []

for grp in large_groups:
    g = ops[ops["comparability_group"] == grp].sort_values("mean_los")
    n_show = min(10, len(g))
    t, b = g.head(n_show), g.tail(n_show)
    all_top_sd.extend(t["sameday_rate"].tolist())
    all_bot_sd.extend(b["sameday_rate"].tolist())
    all_top_ls.extend(t["longstay_rate"].tolist())
    all_bot_ls.extend(b["longstay_rate"].tolist())
    all_top_div.extend(t["proc_diversity"].tolist())
    all_bot_div.extend(b["proc_diversity"].tolist())

for ax, t_vals, b_vals, title, fmt in [
    (axes[0], all_top_sd, all_bot_sd, "Taxa Alta no Mesmo Dia", "{:.0%}"),
    (axes[1], all_top_ls, all_bot_ls, "Taxa Longa Permanência (>7d)", "{:.0%}"),
    (axes[2], all_top_div, all_bot_div, "Diversidade de Procedimentos", "{:.0f}"),
]:
    data = [t_vals, b_vals]
    bp = ax.boxplot(data, labels=["Top 10", "Bottom 10"], patch_artist=True,
                    boxprops=dict(facecolor="#2ecc71", alpha=0.7),
                    medianprops=dict(color="black", lw=2))
    bp["boxes"][1].set_facecolor("#e74c3c")
    ax.set_title(title)
    _, p = stats.mannwhitneyu(t_vals, b_vals, alternative="two-sided")
    ax.annotate(f"p = {p:.4f}", xy=(0.5, 0.95), xycoords="axes fraction",
                ha="center", fontsize=10, fontweight="bold")

plt.suptitle("Fingerprint Operacional: Top 10 vs Bottom 10 (todos os grupos)", fontsize=13, y=1.02)
plt.tight_layout()
save_plot(fig, "04_operational_fingerprint")
plt.show()

print(f"\nAlta mesmo dia: top={np.median(all_top_sd)*100:.1f}% vs bottom={np.median(all_bot_sd)*100:.1f}%")
print(f"Longa permanência: top={np.median(all_top_ls)*100:.1f}% vs bottom={np.median(all_bot_ls)*100:.1f}%")
print(f"Diversidade: top={np.median(all_top_div):.0f} procs vs bottom={np.median(all_bot_div):.0f} procs")

SPEARMAN ρ: métrica operacional vs ranking de LOS (dentro do grupo)
Correlações significativas: 21 de 35

                                     Grupo        Métrica         ρ            p   n
      emergency_dominant · surgical_center  longstay_rate  0.903896 1.944724e-08  21
     emergency_dominant · mixed_procedures  longstay_rate  0.901393 1.979078e-12  32
emergency_dominant · clinical_mgmt_focused  longstay_rate  0.881818 3.301688e-04  11
     emergency_dominant · diagnostic_heavy  longstay_rate  0.843393 1.351554e-39 142
       elective_dominant · surgical_center       uti_rate  0.769029 2.885310e-05  22
       elective_dominant · surgical_center   sameday_rate -0.728967 1.189415e-04  22
       elective_dominant · surgical_center  longstay_rate  0.705481 2.448687e-04  22
      elective_dominant · mixed_procedures migration_rate  0.695334 1.942311e-03  17
      elective_dominant · mixed_procedures  longstay_rate  0.683049 2.509664e-03  17
emergency_dominant · clinical_mgmt_focused  

  Saved: 04_operational_fingerprint.png

Alta mesmo dia: top=20.6% vs bottom=5.6%
Longa permanência: top=1.9% vs bottom=6.8%
Diversidade: top=15 procs vs bottom=17 procs


/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_99217/3514338745.py:58: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=["Top 10", "Bottom 10"], patch_artist=True,
/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_99217/3514338745.py:58: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=["Top 10", "Bottom 10"], patch_artist=True,
/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_99217/3514338745.py:58: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=["Top 10", "Bottom 10"], patch_artist=True,


### 6d. Same-City Case Study — São Paulo Capital

24 hospitals in the same city, same group (Urgência + Diagnóstica), same regulatory environment. LOS ranges from 2.2 to 7.1 days. What explains the gap?

In [13]:
# === 6d. Same-City Case Study: São Paulo ===
from shared import load_cnes_names

names_df = load_cnes_names()
SP_CODE = "355030"
grp_diag = "hospital_geral__emergency_dominant__diagnostic_heavy"

sp_adm = kidney[kidney["MUNIC_MOV"] == SP_CODE]
sp_hosp = sp_adm.groupby("CNES").agg(
    n=("CNES", "count"),
    mean_los=("DIAS_PERM", "mean"),
    sameday_rate=("DIAS_PERM", lambda x: (x == 0).mean()),
    longstay_rate=("DIAS_PERM", lambda x: (x > 7).mean()),
    mean_age=("age", "mean"),
    pct_elderly=("age", lambda x: (x >= 65).mean()),
    pct_surgical=("proc_category", lambda x: (x == "SURGICAL").mean()),
    pct_diagnostic=("proc_category", lambda x: (x == "DIAGNOSTIC").mean()),
    uti_rate=("MARCA_UTI", lambda x: (x != "00").mean() if x.dtype == object else 0),
    proc_diversity=("PROC_REA", "nunique"),
).reset_index()
sp_hosp = sp_hosp[sp_hosp["n"] >= 20]
sp_hosp = sp_hosp.merge(tags[["CNES", "comparability_group"]], on="CNES", how="left")
sp_hosp = sp_hosp.merge(cnes[["CNES", "total_beds", "legal_nature"]], on="CNES", how="left")
sp_hosp = sp_hosp.merge(names_df[["CNES", "nome_fantasia"]], on="CNES", how="left")

sp_diag = sp_hosp[sp_hosp["comparability_group"] == grp_diag].sort_values("mean_los")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. LOS bar chart with hospital names
sp_diag["short_name"] = sp_diag["nome_fantasia"].str.replace("HOSPITAL ", "").str.replace("MUNICIPAL ", "Mun. ").str[:25]
colors = ["#2ecc71" if los <= sp_diag["mean_los"].median() else "#e74c3c" for los in sp_diag["mean_los"]]
axes[0, 0].barh(range(len(sp_diag)), sp_diag["mean_los"], color=colors)
axes[0, 0].set_yticks(range(len(sp_diag)))
axes[0, 0].set_yticklabels(sp_diag["short_name"], fontsize=7)
axes[0, 0].set_xlabel("LOS Médio (dias)")
axes[0, 0].set_title(f"São Paulo · Urg. + Diagnóstica ({len(sp_diag)} hospitais)")
axes[0, 0].axvline(sp_diag["mean_los"].median(), color="black", ls="--", lw=1, label="Mediana")
axes[0, 0].invert_yaxis()

# 2. Same-day rate vs LOS
axes[0, 1].scatter(sp_diag["sameday_rate"] * 100, sp_diag["mean_los"],
                   s=sp_diag["n"] / 5, alpha=0.7, c=colors, edgecolors="black")
axes[0, 1].set_xlabel("Taxa Alta Mesmo Dia (%)")
axes[0, 1].set_ylabel("LOS Médio (dias)")
axes[0, 1].set_title("Alta Mesmo Dia vs LOS")
rho, p = stats.spearmanr(sp_diag["sameday_rate"], sp_diag["mean_los"])
axes[0, 1].annotate(f"ρ = {rho:.2f}, p = {p:.4f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=10)

# 3. Long-stay rate vs LOS
axes[1, 0].scatter(sp_diag["longstay_rate"] * 100, sp_diag["mean_los"],
                   s=sp_diag["n"] / 5, alpha=0.7, c=colors, edgecolors="black")
axes[1, 0].set_xlabel("Taxa Longa Permanência >7d (%)")
axes[1, 0].set_ylabel("LOS Médio (dias)")
axes[1, 0].set_title("Longa Permanência vs LOS")
rho2, p2 = stats.spearmanr(sp_diag["longstay_rate"], sp_diag["mean_los"])
axes[1, 0].annotate(f"ρ = {rho2:.2f}, p = {p2:.4f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=10)

# 4. Beds vs LOS
axes[1, 1].scatter(sp_diag["total_beds"], sp_diag["mean_los"],
                   s=sp_diag["n"] / 5, alpha=0.7, c=colors, edgecolors="black")
axes[1, 1].set_xlabel("Total de Leitos")
axes[1, 1].set_ylabel("LOS Médio (dias)")
axes[1, 1].set_title("Leitos vs LOS")
rho3, p3 = stats.spearmanr(sp_diag["total_beds"], sp_diag["mean_los"])
axes[1, 1].annotate(f"ρ = {rho3:.2f}, p = {p3:.4f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=10)

plt.suptitle("Caso: São Paulo Capital — Mesma Cidade, Mesmo Grupo, LOS Diferente", fontsize=13, y=1.02)
plt.tight_layout()
save_plot(fig, "04_sao_paulo_case_study")
plt.show()

# Print the top 3 vs bottom 3 comparison
top3 = sp_diag.head(3)
bot3 = sp_diag.tail(3)
print("TOP 3 vs BOTTOM 3 — São Paulo · Urgência + Diagnóstica")
print(f"{'':30s} {'TOP 3 (média)':>15s} {'BOTTOM 3 (média)':>18s}")
for col, label, fmt in [
    ("mean_los", "LOS", ".2f"), ("sameday_rate", "Alta mesmo dia", ".1%"),
    ("longstay_rate", "Longa perm. >7d", ".1%"), ("mean_age", "Idade média", ".1f"),
    ("pct_elderly", "% Idosos 65+", ".1%"), ("uti_rate", "% UTI", ".2%"),
    ("total_beds", "Leitos", ".0f"), ("proc_diversity", "Procedimentos", ".0f"),
]:
    t = top3[col].mean()
    b = bot3[col].mean()
    t_str = f"{t:{fmt}}"
    b_str = f"{b:{fmt}}"
    print(f"  {label:30s} {t_str:>15s} {b_str:>18s}")

  Saved: 04_sao_paulo_case_study.png
TOP 3 vs BOTTOM 3 — São Paulo · Urgência + Diagnóstica
                                 TOP 3 (média)   BOTTOM 3 (média)
  LOS                                       2.50               6.01
  Alta mesmo dia                           13.4%               0.7%
  Longa perm. >7d                           5.4%              24.7%
  Idade média                               40.6               41.9
  % Idosos 65+                              7.7%              11.7%
  % UTI                                    0.14%              2.80%
  Leitos                                     214                253
  Procedimentos                                7                  9


### 6e. Legal Nature — Santa Casa vs Público

In [14]:
# === 6e. Legal Nature Analysis ===
hosp_ln = kidney.groupby("CNES").agg(
    n=("CNES", "count"), mean_los=("DIAS_PERM", "mean"),
).reset_index()
hosp_ln = hosp_ln[hosp_ln["n"] >= 20]
hosp_ln = hosp_ln.merge(tags[["CNES", "comparability_group"]], on="CNES", how="left")
hosp_ln = hosp_ln.merge(cnes[["CNES", "legal_nature"]], on="CNES", how="left")

grp_med = hosp_ln.groupby("comparability_group")["mean_los"].median().rename("grp_median")
hosp_ln = hosp_ln.merge(grp_med, on="comparability_group", how="left")
hosp_ln["los_residual"] = hosp_ln["mean_los"] - hosp_ln["grp_median"]

ln_labels = {"assoc_privada": "Santa Casa", "public": "Público",
             "fundacao_privada": "Fundação", "private": "Privado"}
hosp_ln["nature_label"] = hosp_ln["legal_nature"].map(ln_labels)

# Compare Santa Casa vs Público
sc = hosp_ln[hosp_ln["legal_nature"] == "assoc_privada"]["los_residual"]
pub = hosp_ln[hosp_ln["legal_nature"] == "public"]["los_residual"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
data_plot = hosp_ln[hosp_ln["legal_nature"].isin(["assoc_privada", "public"])]
sns.violinplot(data=data_plot, x="nature_label", y="los_residual", ax=axes[0],
               palette={"Santa Casa": "#2ecc71", "Público": "#e74c3c"}, inner="box")
axes[0].axhline(0, color="black", ls="--", lw=1)
axes[0].set_xlabel("")
axes[0].set_ylabel("Resíduo de LOS (vs mediana do grupo)")
axes[0].set_title("Resíduo de LOS por Natureza Jurídica")

_, p_ln = stats.mannwhitneyu(sc, pub, alternative="two-sided")
axes[0].annotate(f"Mann-Whitney p = {p_ln:.2e}", xy=(0.5, 0.95), xycoords="axes fraction",
                 ha="center", fontsize=11, fontweight="bold")

# By group
grp_ln_stats = []
for grp in large_groups:
    g = hosp_ln[hosp_ln["comparability_group"] == grp]
    grp_label = " · ".join(grp.split("__")[1:])
    for nat in ["assoc_privada", "public"]:
        sub = g[g["legal_nature"] == nat]
        if len(sub) >= 3:
            grp_ln_stats.append({
                "Grupo": grp_label, "Natureza": ln_labels[nat],
                "n": len(sub), "Resíduo mediano": sub["los_residual"].median(),
                "% abaixo mediana": (sub["los_residual"] < 0).mean() * 100
            })

grp_ln_df = pd.DataFrame(grp_ln_stats)
pivot = grp_ln_df.pivot(index="Grupo", columns="Natureza", values="Resíduo mediano")
pivot.plot(kind="bar", ax=axes[1], color={"Santa Casa": "#2ecc71", "Público": "#e74c3c"})
axes[1].axhline(0, color="black", ls="--", lw=1)
axes[1].set_ylabel("Resíduo mediano de LOS (dias)")
axes[1].set_title("Resíduo por Grupo e Natureza")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(title="")

plt.tight_layout()
save_plot(fig, "04_legal_nature_effect")
plt.show()

print(f"Santa Casa (n={len(sc)}): resíduo mediano = {sc.median():+.3f}d, {(sc < 0).mean()*100:.1f}% abaixo mediana")
print(f"Público   (n={len(pub)}): resíduo mediano = {pub.median():+.3f}d, {(pub < 0).mean()*100:.1f}% abaixo mediana")
print(f"Mann-Whitney p = {p_ln:.2e}")
print(f"\nEm TODOS os {len(pivot)} grupos, Santa Casas têm resíduo menor que hospitais públicos.")

  Saved: 04_legal_nature_effect.png
Santa Casa (n=159): resíduo mediano = -0.165d, 59.7% abaixo mediana
Público   (n=133): resíduo mediano = +0.253d, 32.3% abaixo mediana
Mann-Whitney p = 2.59e-06

Em TODOS os 7 grupos, Santa Casas têm resíduo menor que hospitais públicos.


/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_99217/2270128780.py:25: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=data_plot, x="nature_label", y="los_residual", ax=axes[0],


## Deep Dive Summary Metrics

In [15]:
# Deep-dive summary metrics
deep_metrics = {
    "patient_mix_age_confounding": f"{len(age_sig)}/{len(large_groups)} groups significant",
    "patient_mix_uti_confounding": f"{len(mix_df[(mix_df['Variável'] == '% UTI') & (mix_df['Significativo'] == 'Sim')])}/{len(large_groups)} groups significant",
    "top_median_beds": np.median(all_top_beds),
    "bottom_median_beds": np.median(all_bot_beds),
    "beds_mannwhitney_p": p_beds,
    "top_median_sameday_pct": np.median(all_top_sd) * 100,
    "bottom_median_sameday_pct": np.median(all_bot_sd) * 100,
    "top_median_longstay_pct": np.median(all_top_ls) * 100,
    "bottom_median_longstay_pct": np.median(all_bot_ls) * 100,
    "longstay_max_spearman_rho": sig_corr[sig_corr["Métrica"] == "longstay_rate"]["ρ"].iloc[0] if len(sig_corr[sig_corr["Métrica"] == "longstay_rate"]) else None,
    "santa_casa_residual_median": sc.median(),
    "publico_residual_median": pub.median(),
    "legal_nature_p": p_ln,
    "sp_case_study_los_range": f"{sp_diag['mean_los'].min():.2f}-{sp_diag['mean_los'].max():.2f}",
}

save_metrics(deep_metrics, "04_deep_dive")
for k, v in deep_metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 04_deep_dive.json
  patient_mix_age_confounding: 0/7 groups significant
  patient_mix_uti_confounding: 3/7 groups significant
  top_median_beds: 110.0
  bottom_median_beds: 212.5
  beds_mannwhitney_p: 4.82317363731566e-08
  top_median_sameday_pct: 20.619483491337036
  bottom_median_sameday_pct: 5.643037022027962
  top_median_longstay_pct: 1.8731098746114498
  bottom_median_longstay_pct: 6.760617025328508
  longstay_max_spearman_rho: 0.9038961038961039
  santa_casa_residual_median: -0.1652109656754508
  publico_residual_median: 0.2534385755516906
  legal_nature_p: 2.5865870462846786e-06
  sp_case_study_los_range: 2.23-7.09
